# LangGraph

## 1) Basics

In [ ]:
from typing import TypedDict    # Pydantic을 사용해도 됨
from langgraph.graph import StateGraph, START, END

In [2]:
class MyState(TypedDict):
    user_input: str
    result: str

In [3]:
def mainagent_node(state: MyState) -> MyState:
    query = state['user_input']
    # llm agent로 query 실행
    
    return {
        'result': f"에이전트의 결과는 1234 입니다."
    }

In [4]:
graph_builder = StateGraph(MyState)

graph_builder.add_node('mainagent', mainagent_node)

graph_builder.add_edge(START, 'mainagent')

graph_builder.add_edge('mainagent', END)

In [5]:
basic_app = graph_builder.compile()

result = basic_app.invoke({'user_input': '안녕하세요'})
result

{'user_input': '안녕하세요', 'result': '에이전트의 결과는 1234 입니다.'}

## 2) Examples

### a) 책 주제 -> 책 제목 LangGraph

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [69]:
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

from collections.abc import Callable
from langgraph.types import Command
from langchain.agents.middleware import ModelFallbackMiddleware, ModelRequest, ModelResponse, wrap_model_call
from langchain_google_genai import ChatGoogleGenerativeAI # gpt-5.4-mini가 되지 않으면 gemini로 바꿈

In [70]:
# 1. State 정의
class BookState(TypedDict):
    topic: str
    title: str

# structured output schema
class BookTitleOutput(BaseModel):
    title: str = Field(description='사용자의 책의 주제를 바탕으로 선정된 제목 1개')

In [71]:
# Fallback model
gemini_fallback = ChatGoogleGenerativeAI(
    model='gemini-3.1-flash',
    temperature=0
)

# Fallback Middleware
@wrap_model_call
def fallback_model_middleware(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse | Command]
) -> ModelResponse | Command:
    try:
        result = handler(request)
        return result
    
    except Exception as e:
        request.model = gemini_fallback
        
        return handler(request)

In [ ]:
# 2. LLM 준비
model = ChatOpenAI(
    model = 'google/gemma-4-26B-A4B-it',
    base_url="http://100.95.34.69:8001/v1",
    api_key='dummy',
)

llm = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(BookTitleOutput),
    system_prompt="""
    당신은 세계에서 제일 유명한 출판사의 메인 기획자입니다.
    사용자가 책의 주제를 입력하면 아래 조건을 만족하는 책의 제목을 1개만 자연스럽게 만들어주세요.
    
    [조건]
    - 짧고 명확하게 작성할 것
    - 너무 과장된 표현은 피할 것
    - 제목만 출력할 것
    - 베스트셀러가 될 제목으로만 만들 것
    - 주제와 제목이 같지 않을 것
    """,
    middleware=[
        fallback_model_middleware,
        ModelFallbackMiddleware(
            "openai:gpt-5.4-mini",
            gemini_fallback,
            'anthropic:claude-sonnet-4-5'
        )
    ]
)


In [73]:
# 3. Node 정의
def make_title_node(state: BookState) -> BookState:
    topic = state['topic']
    
    response = llm.invoke(
        {
            'messages': [
                {
                    'role': 'user',
                    'content': f"책의 주제: {topic}"
                }
            ]
        }
    )
    
    structured_output: BookTitleOutput = response['structured_response']  # {title: '책 제목'}
    
    return {
        'title': structured_output.title
    }

In [74]:
# 4. 그래프 생성
graph_builder = StateGraph(BookState)

# 5. 노드 추가
graph_builder.add_node('make_title', make_title_node)

# 6. 엣지 연결
graph_builder.add_edge(START, 'make_title')
graph_builder.add_edge('make_title', END)

# 7. 컴파일
graph = graph_builder.compile()

# 8. 실행
result = graph.invoke({'topic': '초등학생을 위한 RAG 에이전트 개발'})

In [75]:
print(result)
print("생성된 책 제목:", result["title"])

{'topic': '초등학생을 위한 RAG 에이전트 개발', 'title': '나만의 똑똑한 AI 비서 만들기'}
생성된 책 제목: 나만의 똑똑한 AI 비서 만들기
